# 00 — Freeze Splits & Label Space

**Run this notebook ONCE, then commit `artifacts/splits/`.**  
Nobody re-runs this after the initial freeze.

Steps:
1. Load the full train JSONL
2. Fit and freeze the label space → `artifacts/splits/label_space.json`
3. Run the stratified split → `artifacts/splits/train_idx.npy`, `val_idx.npy`
4. Smoke-test the prediction contract (io roundtrip)
5. Print the per-split stats so we can confirm distributions match

In [ ]:
import sys
sys.path.insert(0, "..")  # make shared/ importable from notebooks/

from shared.config import TRAIN_PATH, TEST_PATH, DEVICE
from shared.data   import load_train_test
from shared.labels import build_label_space, get_mlb
from shared.splits import freeze_split
from shared.io     import roundtrip_check

print(f"Device  : {DEVICE}")

## 1. Load data

In [ ]:
import numpy as np

train_full_df, test_df = load_train_test(TRAIN_PATH, TEST_PATH)

## 2. Freeze the label space

In [ ]:
# This will raise FileExistsError if already frozen — that's intentional.
classes = build_label_space(train_full_df)
print(f"Classes: {classes[:10]} ... ({len(classes)} total)")

## 3. Freeze the train/val split

In [ ]:
mlb, classes, num_classes = get_mlb()
y_full = mlb.transform(train_full_df["rechtsfeitcodes"]).astype("float32")

# This will raise FileExistsError if already frozen — that's intentional.
train_idx, val_idx = freeze_split(train_full_df, y_full)

## 4. Smoke-test the io contract

In [ ]:
roundtrip_check(classes)

## 5. Sanity check — label distribution across splits

In [ ]:
import pandas as pd

train_df = train_full_df.iloc[train_idx].reset_index(drop=True)
val_df   = train_full_df.iloc[val_idx].reset_index(drop=True)

y_train = mlb.transform(train_df["rechtsfeitcodes"]).astype("float32")
y_val   = mlb.transform(val_df["rechtsfeitcodes"]).astype("float32")
y_test  = mlb.transform(test_df["rechtsfeitcodes"]).astype("float32")

stats = pd.DataFrame({
    "split"         : ["train", "val", "test"],
    "docs"          : [len(train_df), len(val_df), len(test_df)],
    "multi_label_%" : [
        f"{(y.sum(axis=1) > 1).mean():.1%}"
        for y in [y_train, y_val, y_test]
    ],
    "zero_support_labels": [
        int((y.sum(axis=0) == 0).sum())
        for y in [y_train, y_val, y_test]
    ],
})
print(stats.to_string(index=False))
print("\n✓ All splits committed. Share artifacts/splits/ with the team.")